In [ ]:
import pandas as pd
import os
from os import path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
DF_metadata = pd.read_csv("../data/processed_data/metadata.tsv", sep='\t', index_col=0)
DF_log_normalizedCounts = pd.read_csv("../data/processed_data/log_normalizedCounts.csv",index_col=0)

## Release Date Analysis

In [ ]:
DF_metadata['ReleaseDate'] = pd.to_datetime(DF_metadata['ReleaseDate'])

In [ ]:
cum_count = DF_metadata['ReleaseDate'].groupby(DF_metadata['ReleaseDate'].dt.year).count().cumsum()

plt.plot(cum_count.index, cum_count)
plt.xlabel('Date Completed')
plt.ylabel('Cumulative Count of Genomes')
plt.title('Cumulative Count of RNA-Seq Profiles Over Time')
plt.xlim([cum_count.index.min(), cum_count.index.max()]) 
plt.show()


## Cluster the Samples

A clustermap is a great way to visualize the global correlations between one sample and all others. The ``global_clustering`` function uses hierarchical clustering to identify specific clusters in the clustermap. The optional arguments are:

* ``threshold``: Threshold used to extract clusters from the hierarchy. To increase the number of clusters, decrease the value of ``threshold``. To decrease the number of clusters, increase the value of ``threshold`` (default: 0.3)
* ``figsize``: A tuple describing the length and width of the final clustermap. A larger figsize can make x and y-axis labels clearer.
* ``xticklabels``: Show NCBI SRA accession numbers on the x-axis
* ``yticklabels``: Show NCBI SRA accession numbers on the y-axis

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.patches as patches

def global_clustering(data, threshold=0.3, xticklabels=False, yticklabels=False, figsize=(9,9)):
    
    # Retrieve clusters using fcluster 
    corr = data.corr()
    corr.fillna(0,inplace=True)
    dist = sch.distance.pdist(corr)
    link = sch.linkage(dist, method='complete')
    clst = pd.DataFrame(index=data.columns)
    clst['cluster'] = sch.fcluster(link, threshold * dist.max(), 'distance')

    # Get colors for each cluster
    cm = plt.cm.get_cmap('tab20')
    cluster_colors = dict(zip(clst.cluster.unique(), cm.colors))
    clst['color'] = clst.cluster.map(cluster_colors)

    print('Number of cluster: ', len(cluster_colors))
    
    legend_items = [patches.Patch(color=c, label=l) for l,c in cluster_colors.items()]
    
    plt.rcParams['figure.facecolor'] = 'white'

    
    clst_map = sns.clustermap(data.corr(), 
                              figsize=figsize, 
                              row_linkage=link, 
                              col_linkage=link, 
                              col_colors=clst.color,
                              yticklabels=yticklabels, 
                              xticklabels=xticklabels,
                              vmin=0, 
                              vmax=1)
    
    legend = clst_map.ax_heatmap.legend(loc='upper left', 
                                        bbox_to_anchor=(1.01,0.85), 
                                        handles=legend_items,
                                        frameon=True)
    
    legend.set_title(title='Clusters',prop={'size':10})
    
    return clst['cluster']

In [ ]:
global_clustering(DF_log_normalizedCounts);

## PCA

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

In [ ]:
pca = PCA()
weights = pd.DataFrame(pca.fit_transform(DF_log_normalizedCounts.T), index = DF_log_normalizedCounts.columns)
components = pd.DataFrame(pca.components_.T, index=DF_log_normalizedCounts.index)

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
cumsum = np.cumsum(pca.explained_variance_ratio_)

dimensionality = np.where(cumsum > .99)[0][0]
print('99% Variance explained with', dimensionality, "components")

ax.plot(cumsum);
ax.axhline(.99, linestyle='dashed');
ax.axvline(dimensionality, linestyle= 'dashed');
ax.set_ylim(0,1);
ax.set_xlim(0,len(weights));
ax.set_title("PCA Explained Variance");
ax.set_xlabel('Number of Dimensions');
ax.set_ylabel('Fraction of Explained Variance');

In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(6,7))

# Make a color map with as many distinct colors as there are projects
projects = DF_metadata['project'].unique()
cmap = cm.get_cmap("tab20", len(projects))  # or "tab20b", "tab20c", "nipy_spectral"

# Map each project to a color
color_dict = {proj: cmap(i) for i, proj in enumerate(projects)}

for cond, group in DF_metadata.groupby('project'):
    ax.scatter(weights.loc[group.index, 0],
               weights.loc[group.index, 1],
               label=cond, alpha=.6,
               color=color_dict[cond])

ax.set_title("PC1 vs PC2")
ax.set_xlabel('PC1: %.2f%%'%(pca.explained_variance_ratio_[0]*100))
ax.set_ylabel('PC2: %.2f%%'%(pca.explained_variance_ratio_[1]*100))

ax.legend(
    ncol=1,                         # spread into 2 columns
    loc='upper right', 
    bbox_to_anchor=(1.75, 1), 
    fontsize=10,                     # smaller text
    markerscale=0.7,                 # smaller legend markers
    handletextpad=0.2,              # less padding between marker & text
    columnspacing=0.2,
    labelspacing=0.2                
)
plt.savefig('figures/images/fig1_pca.svg', format = 'svg')
plt.show()

In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(10,10))

# Make a color map with as many distinct colors as there are projects
projects = DF_metadata['project'].unique()
cmap = cm.get_cmap("tab20", len(projects))  # or "tab20b", "tab20c", "nipy_spectral"

# Map each project to a color
color_dict = {proj: cmap(i) for i, proj in enumerate(projects)}

for cond, group in DF_metadata.groupby('project'):
    ax.scatter(weights.loc[group.index, 0],
               weights.loc[group.index, 1],
               label=cond, alpha=.6,
               color=color_dict[cond])

ax.set_title("PC1 vs PC2")
ax.set_xlabel('PC1: %.2f%%'%(pca.explained_variance_ratio_[0]*100))
ax.set_ylabel('PC2: %.2f%%'%(pca.explained_variance_ratio_[1]*100))

ax.legend(
    ncol=1,                         # spread into 2 columns
    loc='upper right', 
    bbox_to_anchor=(1.75, 1), 
    fontsize=10,                     # smaller text
    markerscale=0.7,                 # smaller legend markers
    handletextpad=0.2,              # less padding between marker & text
    columnspacing=0.2,
    labelspacing=0.2                
)
plt.savefig('figures/images/fig1_pca_full_size.svg', format = 'svg')
plt.show()